# <center> <font color="#0036a3">Maestría en Inteligencia Artificial Aplicada (MNA)</font> </center>

<center>

[![Materia](https://img.shields.io/badge/MATERIA-PROYECTO_INTEGRADOR-E0A800?style=for-the-badge&logoColor=white)](https://tec.mx)

</center>

<center>

[![Python](https://img.shields.io/badge/Python-3776AB?style=flat-square&logo=python&logoColor=white)](https://www.python.org/)
[![Jupyter](https://img.shields.io/badge/Jupyter-F37626?style=flat-square&logo=jupyter&logoColor=white)](https://jupyter.org/)
[![PyTorch](https://img.shields.io/badge/PyTorch-EE4C2C?style=flat-square&logo=pytorch&logoColor=white)](https://pytorch.org/)
[![OpenCV](https://img.shields.io/badge/OpenCV-5C3EE8?style=flat-square&logo=opencv&logoColor=white)](https://opencv.org/)
[![GitHub](https://img.shields.io/badge/Repo-GitHub-181717?style=flat-square&logo=github&logoColor=white)](https://github.com/jmtoral/proyecto_integrador_52)

</center>

## **<font color="#0036a3">Avance 4 — Photometric Tracking: Visualización sobre SCARED</font>**

### **<font color="#E0A800">Proyecto Integrador — TC5035.10</font>**

---

## **<center> <font color="#0036a3">Equipo 52</font> </center>**

<table style="border-collapse:collapse; width:60%; margin:auto;">
  <tr>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/elda.jpg" width="80" height="80" style="border-radius:50%; object-fit:cover; object-position:center top;"><br>
      <strong>Elda Morales</strong><br><small>A00449074</small>
    </td>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/mpgc.jpg" width="80" height="80" style="border-radius:50%; object-fit:cover; object-position:center top;"><br>
      <strong>María Paula Gutiérrez</strong><br><small>A01747706</small>
    </td>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/jmtc_n.jpg" width="80" height="80" style="border-radius:50%; object-fit:cover; object-position:center top;"><br>
      <strong>José Manuel Toral</strong><br><small>A01122243</small>
    </td>
  </tr>
</table>

---

### **Objetivos de este notebook**

Replicar la visualización de **Photometric Tracking** de Recasens et al. (2021) usando los datos SCARED:

1. Cargar el `rgb.mp4` y las poses GT (`frame_data.tar.gz`) de un keyframe de SCARED
2. Predecir el depth map del keyframe con Endo-Depth (igual que en Avance 3)
3. Para cada frame del video, **reproyectar el keyframe** al punto de vista del frame usando la pose GT y el depth map
4. Calcular el **error fotométrico** entre frame actual y keyframe reproyectado
5. Generar la **animación de 5 columnas** replicando el GIF del paper
6. Explorar si la **corrección de iluminación** (Retinex) reduce el error fotométrico

> Recasens, D., Lamarca, J., Fácil, J. M., Montiel, J. M. M., & Civera, J. (2021). Endo-Depth-and-Motion: Reconstruction and Tracking in Endoscopic Videos using Depth Networks and Photometric Constraints. *arXiv:2103.16525 [cs.CV]*. https://doi.org/10.48550/arXiv.2103.16525

---
## 0. Configuración

### 0.1 Rutas

| Recurso | Ruta en Drive |
|---|---|
| `scared_raw/` | `MyDrive/proyecto_integrador/scared_raw/` |
| `endo_depth_weights/` | `MyDrive/proyecto_integrador/endo_depth_weights/` |
| `Endo-Depth-and-Motion/` | `MyDrive/proyecto_integrador/Endo-Depth-and-Motion/` |
| `avance4_outputs/` | `MyDrive/proyecto_integrador/avance4_outputs/` |

### 0.2 Dataset seleccionado

Usamos `dataset_1 / keyframe_1` porque es el primer dataset disponible y el keyframe con más frames de video (197 frames a 25 FPS ≈ 8 segundos de movimiento real del endoscopio).

In [ ]:
import subprocess, sys

# Solo instalar lo que Colab no trae por defecto
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tifffile"])

import torch, tifffile, cv2, numpy as np
print(f"torch    : {torch.__version__}")
print(f"tifffile : {tifffile.__version__}")
print(f"opencv   : {cv2.__version__}")
print(f"CUDA OK  : {torch.cuda.is_available()}")

In [ ]:
from pathlib import Path
import sys

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE        = Path("/content/drive/MyDrive/proyecto_integrador")
    MODEL_PATH  = BASE / "endo_depth_weights"
    SCARED_ROOT = BASE / "scared_raw"
    EDAM_PATH   = BASE / "Endo-Depth-and-Motion"
    OUT_DIR     = BASE / "avance4_outputs"
else:
    MODEL_PATH  = Path("E:/endo_depth_weights")
    SCARED_ROOT = Path("D:/Proyecto_Integrador/Corrreccion_Luz/data/scared_raw")
    EDAM_PATH   = Path("E:/Endo-Depth-and-Motion")
    OUT_DIR     = Path("../outcomes/avance4")

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Dataset a analizar
DATASET_ID  = "dataset_1"
KEYFRAME_ID = "keyframe_1"

# Submuestreo de frames para la animación (1 = todos, 4 = cada 4to)
FRAME_STEP = 4

print(f"Entorno  : {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Dataset  : {DATASET_ID} / {KEYFRAME_ID}")
print(f"OUT_DIR  : {OUT_DIR}")

---
## 1. Cargar Endo-Depth

Mismo modelo que en Avance 3: ResNet18 encoder + DepthDecoder, pesos de Hamlyn.

In [ ]:
import sys
import torch
import numpy as np

sys.path.insert(0, str(EDAM_PATH / "apps" / "depth_estimate"))
from resnet_encoder import ResnetEncoder
from depth_decoder import DepthDecoder

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")

encoder = ResnetEncoder(18, False)
loaded_enc = torch.load(MODEL_PATH / "encoder.pth", map_location=DEVICE)
FEED_HEIGHT = loaded_enc["height"]
FEED_WIDTH  = loaded_enc["width"]

filtered_enc = {k: v for k, v in loaded_enc.items() if k in encoder.state_dict()}
encoder.load_state_dict(filtered_enc)
encoder.to(DEVICE).eval()

depth_decoder = DepthDecoder(num_ch_enc=encoder.num_ch_enc, scales=range(4))
loaded_dec = torch.load(MODEL_PATH / "depth.pth", map_location=DEVICE)
depth_decoder.load_state_dict(loaded_dec)
depth_decoder.to(DEVICE).eval()

print(f"Resolución del modelo: {FEED_HEIGHT}×{FEED_WIDTH}")
print("Modelo cargado ✓")

---
## 2. Cargar keyframe, video y poses

### Estructura del `rgb.mp4`

El video es **estéreo apilado verticalmente**: resolución 1280×2048, donde:
- Mitad superior (filas 0–1023): cámara **izquierda** (la misma que `Left_Image.png`)
- Mitad inferior (filas 1024–2047): cámara **derecha**

Usamos solo el canal izquierdo para ser consistentes con el Avance 3.

### Estructura de `frame_data.tar.gz`

197 archivos JSON (`frame_data000000.json` … `frame_data000196.json`), uno por frame del video. Cada JSON contiene:
- `camera-pose`: matriz **4×4 homogénea** con la pose de la cámara en el frame (R|t en la última columna, escala en metros)
- `camera-calibration.KL`: matriz intrínseca 3×3 de la cámara izquierda

In [ ]:
import zipfile, tarfile, io, json, cv2, tifffile
import numpy as np

zip_path = SCARED_ROOT / f"{DATASET_ID}.zip"

def load_from_zip(zip_path, inner_path):
    with zipfile.ZipFile(zip_path) as z:
        with z.open(inner_path) as f:
            return f.read()

# ── Keyframe RGB ──────────────────────────────────────────────────────────────
kf_prefix = f"{DATASET_ID}/{KEYFRAME_ID}"
img_bytes = load_from_zip(zip_path, f"{kf_prefix}/Left_Image.png")
buf = np.frombuffer(img_bytes, np.uint8)
keyframe_rgb = cv2.cvtColor(cv2.imdecode(buf, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)

# ── Keyframe depth GT ────────────────────────────────────────────────────────
tiff_bytes = load_from_zip(zip_path, f"{kf_prefix}/left_depth_map.tiff")
tiff = tifffile.imread(io.BytesIO(tiff_bytes))
depth_gt = tiff[..., 2].astype(np.float32)   # canal Z en mm
depth_gt[depth_gt <= 0] = np.nan

# ── Video: guardar MP4 temporalmente, leer solo poses y metadatos ─────────────
# No cargamos all_frames en memoria — el video se lee frame a frame en la animación
video_bytes = load_from_zip(zip_path, f"{kf_prefix}/data/rgb.mp4")
tmp_mp4 = OUT_DIR / "_tmp_video.mp4"
tmp_mp4.write_bytes(video_bytes)
del video_bytes  # liberar ~30 MB

# Contar frames y obtener resolución sin cargar todo
cap = cv2.VideoCapture(str(tmp_mp4))
N_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
ret, frame0 = cap.read()
FRAME_H = frame0.shape[0] // 2
FRAME_W = frame0.shape[1]
cap.release()

print(f"Video: {N_FRAMES} frames, resolución izquierda={FRAME_H}×{FRAME_W}")

# ── Poses GT ─────────────────────────────────────────────────────────────────
fd_bytes = load_from_zip(zip_path, f"{kf_prefix}/data/frame_data.tar.gz")
with tarfile.open(fileobj=io.BytesIO(fd_bytes)) as t:
    poses = []
    K = None
    for i in range(N_FRAMES):
        name = f"frame_data{i:06d}.json"
        try:
            d = json.load(t.extractfile(name))
        except KeyError:
            break
        poses.append(np.array(d["camera-pose"], dtype=np.float64))
        if K is None:
            K = np.array(d["camera-calibration"]["KL"], dtype=np.float64)

# Frame 0 como keyframe de referencia para ORB (solo guardamos este)
cap = cv2.VideoCapture(str(tmp_mp4))
ret, frame0 = cap.read()
cap.release()
frame0_left = cv2.cvtColor(frame0[:FRAME_H], cv2.COLOR_BGR2RGB)

print(f"Keyframe    : {keyframe_rgb.shape}  dtype={keyframe_rgb.dtype}")
print(f"Depth GT    : {depth_gt.shape}  rango=[{np.nanmin(depth_gt):.1f}, {np.nanmax(depth_gt):.1f}] mm")
print(f"N frames    : {N_FRAMES}  (leyendo frame a frame para ahorrar RAM)")
print(f"Poses GT    : {len(poses)} matrices 4×4")
print(f"K (left)    :\n{K}")

---
## 3. Predecir depth map del keyframe con Endo-Depth

Usamos el mismo pipeline de inferencia del Avance 3. Aplicamos median scaling con el GT disponible para convertir a mm.

In [ ]:
import torch.nn.functional as F
import PIL.Image as pil
from torchvision import transforms
import matplotlib.pyplot as plt

def predict_depth(img_rgb, encoder, decoder, feed_h, feed_w, device):
    H, W = img_rgb.shape[:2]
    input_pil = pil.fromarray(img_rgb).resize((feed_w, feed_h), pil.LANCZOS)
    input_t = transforms.ToTensor()(input_pil).unsqueeze(0).to(device)
    with torch.no_grad():
        features = encoder(input_t)
        outputs  = decoder(features)
    disp = outputs[("disp", 0)]
    disp_full = F.interpolate(disp, (H, W), mode="bilinear", align_corners=False)
    disp_np = disp_full.squeeze().cpu().numpy()
    min_disp, max_disp = 1.0/100.0, 1.0/0.1
    scaled = min_disp + (max_disp - min_disp) * disp_np
    return 1.0 / scaled

depth_rel = predict_depth(
    keyframe_rgb, encoder, depth_decoder, FEED_HEIGHT, FEED_WIDTH, DEVICE)

# Median scaling con GT
CAP_MM = 150.0
valid = (~np.isnan(depth_gt)) & (depth_gt > 0) & (depth_gt < CAP_MM)
scale = np.median(depth_gt[valid]) / (np.median(depth_rel[valid]) + 1e-8)
depth_pred_mm = depth_rel * scale

print(f"Scale factor    : {scale:.4f}")
print(f"Depth pred rango: [{depth_pred_mm.min():.1f}, {depth_pred_mm.max():.1f}] mm")

# Visualización diagnóstica
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].imshow(keyframe_rgb)
axes[0].set_title("Keyframe (Left_Image.png)", fontsize=11)
axes[0].axis("off")

im = axes[1].imshow(depth_pred_mm, cmap="jet", vmin=0,
                    vmax=np.nanpercentile(depth_pred_mm, 98))
axes[1].set_title("Depth Endo-Depth (mm)", fontsize=11)
axes[1].axis("off")
plt.colorbar(im, ax=axes[1], label="mm")

im2 = axes[2].imshow(depth_gt, cmap="jet", vmin=0,
                     vmax=np.nanpercentile(depth_gt, 98))
axes[2].set_title("Depth GT (luz estructurada)", fontsize=11)
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], label="mm")

plt.suptitle(f"{DATASET_ID}/{KEYFRAME_ID} — Comparativa depth", fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / "avance4_depth_comparativa.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 4. Reproyección fotométrica

### Enfoque: homografía estimada por feature matching

Las poses de `frame_data.tar.gz` del dataset SCARED describen la posición del **escáner de luz estructurada** durante la captura del keyframe, **no** el movimiento del endoscopio a lo largo del video `rgb.mp4`. El desplazamiento entre frame 0 y frame 10 es ~2.88 unidades de pose, equivalente a ~8× la distancia cámara-tejido — físicamente imposible para movimiento endoscópico a 25 fps.

Por esto, estimamos la transformación entre keyframe y cada frame del video usando **ORB + RANSAC → homografía 2D**:

1. Detectar keypoints ORB en keyframe y frame actual
2. Matching con BFMatcher + filtro de ratio de Lowe (0.75)
3. `cv2.findHomography` con RANSAC para estimar H (8 DOF)
4. `cv2.warpPerspective` para reproyectar el keyframe al punto de vista del frame actual

La homografía es una buena aproximación cuando la escena es aproximadamente plana (tejido endoscópico a corta distancia) y el movimiento es predominantemente translacional/rotacional en plano.

### Error fotométrico

$$\text{error} = \frac{1}{|\Omega|} \sum_{p \in \Omega} \|I_t(p) - \hat{I}_{k \to t}(p)\|_1$$

donde $\hat{I}_{k \to t}$ es el keyframe reproyectado vía homografía al frame $t$, y $\Omega$ es la región válida (sin bordes negros de warp).

In [ ]:
import cv2
import numpy as np

def read_frame(video_path, idx):
    """Lee un único frame del video (canal izquierdo) sin cargar todo en memoria."""
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        return None
    h = frame.shape[0] // 2
    return cv2.cvtColor(frame[:h], cv2.COLOR_BGR2RGB)

# ── ORB sobre el keyframe (se computa una sola vez) ──────────────────────────
orb = cv2.ORB_create(nfeatures=2000)
kf_gray = cv2.cvtColor(keyframe_rgb, cv2.COLOR_RGB2GRAY)
kp_kf, des_kf = orb.detectAndCompute(kf_gray, None)
matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)

def reproject_keyframe_homography(keyframe_rgb, frame_rgb):
    H_img, W_img = keyframe_rgb.shape[:2]
    gray = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2GRAY)
    kp_fr, des_fr = orb.detectAndCompute(gray, None)

    if des_fr is None or len(kp_fr) < 10:
        return np.zeros_like(keyframe_rgb), np.zeros((H_img, W_img), bool), None

    matches = matcher.knnMatch(des_kf, des_fr, k=2)
    good = [m for m, n in matches if m.distance < 0.75 * n.distance]

    if len(good) < 10:
        return np.zeros_like(keyframe_rgb), np.zeros((H_img, W_img), bool), None

    src_pts = np.float32([kp_kf[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp_fr[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    H_mat, _ = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 3.0)

    if H_mat is None:
        return np.zeros_like(keyframe_rgb), np.zeros((H_img, W_img), bool), None

    reprojected = cv2.warpPerspective(keyframe_rgb, H_mat, (W_img, H_img),
                                      flags=cv2.INTER_LINEAR,
                                      borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    ones = np.ones((H_img, W_img), dtype=np.uint8) * 255
    mask_warp = cv2.warpPerspective(ones, H_mat, (W_img, H_img),
                                    flags=cv2.INTER_NEAREST,
                                    borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    return reprojected, mask_warp > 128, H_mat


# ── Prueba diagnóstica con frame 10 ──────────────────────────────────────────
import matplotlib.pyplot as plt

test_frame_idx = 10
actual = read_frame(tmp_mp4, test_frame_idx)
reproj, mask, H_mat = reproject_keyframe_homography(keyframe_rgb, actual)

error = np.abs(actual.astype(np.float32) - reproj.astype(np.float32)).mean(axis=2)
error[~mask] = np.nan

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, img, title in zip(axes,
    [keyframe_rgb, depth_pred_mm, actual, reproj, error],
    ["Keyframe", "Depth", "Actual", "Reprojected", "Error"]):
    if title == "Depth":
        ax.imshow(img, cmap="jet", vmin=0, vmax=np.nanpercentile(depth_pred_mm, 98))
    elif title == "Error":
        ax.imshow(img, cmap="hot", vmin=0, vmax=50)
    elif title in ("Actual", "Reprojected"):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), cmap="gray")
    else:
        ax.imshow(img)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.axis("off")

plt.suptitle(f"Reproyección por homografía ORB — frame {test_frame_idx}", fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / "avance4_reproject_test.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Error fotométrico medio: {np.nanmean(error):.2f} DN")
print(f"Cobertura             : {mask.mean()*100:.1f}%")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

test_frame_idx = 10
actual = read_frame(tmp_mp4, test_frame_idx)

reproj_pred, mask_pred, _ = reproject_keyframe_homography(keyframe_rgb, actual)
err_pred = np.abs(actual.astype(np.float32) - reproj_pred.astype(np.float32)).mean(axis=2)
err_pred[~mask_pred] = np.nan

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, img, title in zip(axes,
    [keyframe_rgb, depth_pred_mm, actual,
     cv2.cvtColor(reproj_pred, cv2.COLOR_RGB2GRAY), err_pred],
    ["Keyframe", "Depth Endo-Depth", "Actual", "Reprojected", "Error abs"]):
    if title == "Depth Endo-Depth":
        ax.imshow(img, cmap="jet", vmin=0, vmax=np.nanpercentile(depth_pred_mm, 98))
    elif title == "Error abs":
        ax.imshow(img, cmap="hot", vmin=0, vmax=50)
    elif title in ("Actual", "Reprojected"):
        ax.imshow(img, cmap="gray")
    else:
        ax.imshow(img)
    ax.set_title(title, fontsize=10)
    ax.axis("off")

plt.suptitle(f"Reproyección fotométrica (homografía ORB) — frame {test_frame_idx}",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "avance4_gt_vs_pred_reproject.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Error fotométrico medio (frame {test_frame_idx}): {np.nanmean(err_pred):.2f} DN")

---
## 5. Animación completa — replicando el GIF de Endo-Depth-and-Motion

Generamos un frame por cada `FRAME_STEP` frames del video, con las 5 columnas del paper:
**Keyframe | Depth | Actual | Reprojected | Error**

El resultado se guarda como GIF y como MP4.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import PIL.Image as pil
from IPython.display import Image as IPImage, display

frame_indices = list(range(0, N_FRAMES, FRAME_STEP))
print(f"Generando animación: {len(frame_indices)} frames (FRAME_STEP={FRAME_STEP})")

depth_vmax = float(np.nanpercentile(depth_pred_mm, 98))
# Depth coloreado como imagen fija (no cambia entre frames)
depth_norm = np.clip(depth_pred_mm / depth_vmax, 0, 1)
depth_rgba = (cm.jet(depth_norm)[:, :, :3] * 255).astype(np.uint8)

mean_errors = []
gif_frames  = []

# Leer el video secuencialmente (más eficiente que seek por frame)
cap = cv2.VideoCapture(str(tmp_mp4))
frame_idx = 0
target_set = set(frame_indices)

while True:
    ret, raw = cap.read()
    if not ret:
        break
    if frame_idx not in target_set:
        frame_idx += 1
        continue

    h = raw.shape[0] // 2
    actual = cv2.cvtColor(raw[:h], cv2.COLOR_BGR2RGB)
    reproj, mask, _ = reproject_keyframe_homography(keyframe_rgb, actual)

    err = np.abs(actual.astype(np.float32) - reproj.astype(np.float32)).mean(axis=2)
    err[~mask] = np.nan
    mean_err = float(np.nanmean(err))
    mean_errors.append(mean_err)

    # Componer panel de 5 columnas directamente como imagen PIL
    # Redimensionar a altura fija para el GIF (ahorrar tamaño)
    GIF_H, GIF_W = 256, 320
    cols = [
        keyframe_rgb,
        depth_rgba,
        cv2.cvtColor(actual,  cv2.COLOR_RGB2GRAY),
        cv2.cvtColor(reproj,  cv2.COLOR_RGB2GRAY),
        None,  # error — se genera abajo
    ]
    err_norm = np.clip(err / 50.0, 0, 1)
    err_rgb = (cm.hot(np.nan_to_num(err_norm, nan=0))[:, :, :3] * 255).astype(np.uint8)
    cols[4] = err_rgb

    panels = []
    for c in cols:
        if c.ndim == 2:
            c = np.stack([c]*3, axis=2)
        p = pil.fromarray(c).resize((GIF_W, GIF_H), pil.LANCZOS)
        panels.append(np.array(p))

    row = np.concatenate(panels, axis=1)
    gif_frames.append(pil.fromarray(row))

    frame_idx += 1

cap.release()

print(f"Error fotométrico — min: {min(mean_errors):.2f}  max: {max(mean_errors):.2f}  media: {np.mean(mean_errors):.2f}")

# Guardar GIF
gif_path = OUT_DIR / "avance4_photometric_tracking.gif"
gif_frames[0].save(
    str(gif_path),
    save_all=True,
    append_images=gif_frames[1:],
    duration=100,
    loop=0,
    optimize=True,
)
print(f"GIF guardado: {gif_path}  ({gif_path.stat().st_size / 1e6:.1f} MB)")
del gif_frames  # liberar RAM

display(IPImage(str(gif_path)))

---
## 6. Error fotométrico a lo largo del video

El error fotométrico medio por frame nos dice cómo varía la calidad de la reproyección conforme el endoscopio se aleja del keyframe. Esperamos que aumente monotónicamente: a mayor desplazamiento, mayor error.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(frame_indices, mean_errors, color="#E0A800", linewidth=2)
ax.set_xlabel("Frame del video")
ax.set_ylabel("Error fotométrico medio (DN)")
ax.set_title("Error fotométrico vs. posición en el video", fontsize=12)
ax.grid(alpha=0.3)

plt.suptitle(f"Análisis del error fotométrico — {DATASET_ID}/{KEYFRAME_ID}",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "avance4_error_fotometrico.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Error mín / máx : {min(mean_errors):.2f} / {max(mean_errors):.2f} DN")
print(f"Error medio     : {np.mean(mean_errors):.2f} DN")

---
## 7. Visualización 3D de la nube de puntos

Reconstruimos la nube de puntos del keyframe en 3D usando el depth map predicho por Endo-Depth y las intrínsegas `K`. Cada punto se colorea con el RGB original del keyframe, replicando conceptualmente la reconstrucción 3D del paper.

La trayectoria de cámara no se dibuja aquí porque la homografía produce transformaciones independientes (keyframe→frame_t), no una trayectoria acumulativa coherente — eso requeriría odometría visual incremental.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import cv2

# ── Nube de puntos del keyframe ───────────────────────────────────────────────
H_img, W_img = depth_pred_mm.shape
fx, fy = K[0, 0], K[1, 1]
cx, cy = K[0, 2], K[1, 2]

u = np.arange(W_img, dtype=np.float32)
v = np.arange(H_img, dtype=np.float32)
uu, vv = np.meshgrid(u, v)

Z = depth_pred_mm.astype(np.float32)
valid = np.isfinite(Z) & (Z > 0) & (Z < 120)

X_all = ((uu - cx) * Z / fx)
Y_all = ((vv - cy) * Z / fy)

# ── Vista 1: 3D con coloreado por profundidad (más legible que RGB) ───────────
step = 6
Xv = X_all[valid][::step]
Yv = Y_all[valid][::step]
Zv = Z[valid][::step]

depth_norm = (Zv - Zv.min()) / (Zv.max() - Zv.min() + 1e-8)
colors_depth = cm.turbo(depth_norm)

fig = plt.figure(figsize=(16, 5))

ax1 = fig.add_subplot(1, 3, 1, projection="3d")
sc = ax1.scatter(Xv, Zv, -Yv, c=depth_norm, cmap="turbo", s=0.5, alpha=0.6, linewidths=0)
ax1.set_xlabel("X (mm)", fontsize=8)
ax1.set_ylabel("Z / profundidad (mm)", fontsize=8)
ax1.set_zlabel("-Y (mm)", fontsize=8)
ax1.set_title("3D — coloreado por profundidad", fontsize=10)
ax1.view_init(elev=20, azim=-55)
plt.colorbar(sc, ax=ax1, label="prof. norm.", shrink=0.5, pad=0.1)

# ── Vista 2: proyección frontal (X vs Y) coloreada por profundidad ────────────
ax2 = fig.add_subplot(1, 3, 2)
sc2 = ax2.scatter(Xv, -Yv, c=depth_norm, cmap="turbo", s=0.3, alpha=0.7, linewidths=0)
ax2.set_xlabel("X (mm)")
ax2.set_ylabel("-Y (mm)")
ax2.set_title("Vista frontal — proyección XY", fontsize=10)
ax2.set_aspect("equal")
plt.colorbar(sc2, ax=ax2, label="prof. norm.", shrink=0.8)

# ── Vista 3: depth map como imagen 2D (más nítida de todas) ──────────────────
ax3 = fig.add_subplot(1, 3, 3)
depth_vis = np.where(valid, Z, np.nan)
im = ax3.imshow(depth_vis, cmap="turbo", vmin=np.nanpercentile(Z[valid], 2),
                vmax=np.nanpercentile(Z[valid], 98))
ax3.set_title("Depth map Endo-Depth (mm)", fontsize=10)
ax3.axis("off")
plt.colorbar(im, ax=ax3, label="mm", shrink=0.8)

fig.suptitle("Reconstrucción 3D del keyframe — Endo-Depth sobre SCARED", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "avance4_3d_pointcloud.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Nube de puntos: {len(Xv):,} puntos  |  rango Z: [{Zv.min():.1f}, {Zv.max():.1f}] mm")

---
## 8. ¿Retinex reduce el error fotométrico?

Hipótesis de este avance: si aplicamos Retinex al keyframe y a cada frame actual antes de calcular el error fotométrico, ¿el error baja? Esto extendería el resultado del Avance 3 (Retinex mejora AbsRel vs. GT) a un escenario de tracking en movimiento.

Comparamos dos condiciones:
- **none**: keyframe original vs. frame actual original
- **retinex**: keyframe Retinex vs. frame actual Retinex (comparación justa)

Implementamos Single-Scale Retinex (SSR) con σ=30, siguiendo la formulación de Rahman et al. (2004):

$$R_{SSR}(x,y) = \log I(x,y) - \log \left[ G_\sigma * I(x,y) \right]$$

> Rahman, Z., Jobson, D. J., & Woodell, G. A. (2004). Retinex processing for automatic image enhancement. *Journal of Electronic Imaging*, 13(1), 100–110. https://doi.org/10.1117/1.1636183

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def correct_retinex(img_rgb, sigma=30):
    img_f = img_rgb.astype(np.float32) + 1.0
    result = np.zeros_like(img_f)
    for c in range(3):
        blur = cv2.GaussianBlur(img_f[:, :, c], (0, 0), sigma)
        result[:, :, c] = np.log(img_f[:, :, c]) - np.log(blur + 1.0)
    result -= result.min()
    return (result / (result.max() + 1e-8) * 255).astype(np.uint8)

keyframe_retinex = correct_retinex(keyframe_rgb)

errors_none    = []
errors_retinex = []

# Leer video secuencialmente
cap = cv2.VideoCapture(str(tmp_mp4))
frame_idx = 0
target_set = set(frame_indices)

while True:
    ret, raw = cap.read()
    if not ret:
        break
    if frame_idx not in target_set:
        frame_idx += 1
        continue

    h = raw.shape[0] // 2
    actual = cv2.cvtColor(raw[:h], cv2.COLOR_BGR2RGB)
    actual_retinex = correct_retinex(actual)

    reproj_n, mask_n, _ = reproject_keyframe_homography(keyframe_rgb, actual)
    err_n = np.abs(actual.astype(np.float32) - reproj_n.astype(np.float32)).mean(axis=2)
    err_n[~mask_n] = np.nan
    errors_none.append(float(np.nanmean(err_n)))

    reproj_r, mask_r, _ = reproject_keyframe_homography(keyframe_retinex, actual_retinex)
    err_r = np.abs(actual_retinex.astype(np.float32) - reproj_r.astype(np.float32)).mean(axis=2)
    err_r[~mask_r] = np.nan
    errors_retinex.append(float(np.nanmean(err_r)))

    frame_idx += 1

cap.release()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(frame_indices, errors_none,    label="Sin corrección (none)", color="#2766CB", linewidth=2)
ax.plot(frame_indices, errors_retinex, label="Retinex SSR (σ=30)",
        color="#E0A800", linewidth=2, linestyle="--")
ax.set_xlabel("Frame del video")
ax.set_ylabel("Error fotométrico medio (DN)")
ax.set_title("Impacto de Retinex sobre el error fotométrico durante el tracking",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

mean_none    = np.nanmean(errors_none)
mean_retinex = np.nanmean(errors_retinex)
mejora = (mean_none - mean_retinex) / mean_none * 100
ax.text(0.02, 0.90,
        f"Media none={mean_none:.2f}  retinex={mean_retinex:.2f}  mejora={mejora:+.1f}%",
        transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))

plt.tight_layout()
plt.savefig(OUT_DIR / "avance4_retinex_vs_none.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Error medio none    : {mean_none:.2f} DN")
print(f"Error medio retinex : {mean_retinex:.2f} DN")
print(f"Diferencia          : {mejora:+.1f}%")

---
## 9. Conclusiones

### Lo que replicamos

La visualización de **Photometric Tracking** del paper Endo-Depth-and-Motion: keyframe estático con depth map predicho por Endo-Depth, reproyectado fotométricamente a lo largo de la secuencia de video del `rgb.mp4` usando homografía ORB+RANSAC para estimar el movimiento real del endoscopio.

### Hallazgo principal

**Retinex SSR (σ=30) reduce el error fotométrico en un 29.8%** (26.44 → 18.57 DN) a lo largo de toda la secuencia. La mejora es consistente en todos los frames y especialmente pronunciada cuando el endoscopio se aleja del keyframe (frames 100–196), donde las variaciones de iluminación especular son más severas.

### Por qué las poses GT no se usaron

Las poses de `frame_data.tar.gz` del dataset SCARED describen la geometría del **escáner de luz estructurada**, no el movimiento del endoscopio en `rgb.mp4`. El desplazamiento entre frames consecutivos (~0.3 unidades) equivale a ~8× la distancia cámara-tejido — físicamente imposible a 25 fps. Por esto se usó homografía ORB+RANSAC para estimar la transformación real entre frames.

### Implicaciones para el proyecto

El error fotométrico durante tracking es una métrica complementaria al AbsRel del Avance 3: mide si la corrección de iluminación se traduce en mejor rastreo visual en video real. El resultado de +29.8% confirma la hipótesis del proyecto también en el escenario dinámico más relevante clínicamente.

## Referencias

- Recasens, D., Lamarca, J., Fácil, J. M., Montiel, J. M. M., & Civera, J. (2021). Endo-Depth-and-Motion: Reconstruction and Tracking in Endoscopic Videos using Depth Networks and Photometric Constraints. *arXiv:2103.16525 [cs.CV]*. https://doi.org/10.48550/arXiv.2103.16525
- Allan, M., et al. (2021). Stereo Correspondence and Reconstruction of Endoscopic Data Challenge. *arXiv:2101.01133*.
- Rahman, Z., Jobson, D. J., & Woodell, G. A. (2004). Retinex processing for automatic image enhancement. *Journal of Electronic Imaging*, 13(1), 100–110. https://doi.org/10.1117/1.1636183
- Land, E. H., & McCann, J. J. (1971). Lightness and retinex theory. *Journal of the Optical Society of America*, 61(1), 1–11.